In [ ]:
import os
import pandas as pd
import boto3
from tqdm import tqdm
import pickle
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')

### Functions

In [ ]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### Constants

In [ ]:
# constants
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_variant = 'noPTImodel7'
str_dirname_output = './output'

### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant dir

In [ ]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Load preprocessed data

In [ ]:
# load preprocessed data
print('Loading data...')
list_str_df = [
    'train',
    'valid',
    'test',
]
list_df = []
for str_df in list_str_df:
    str_filename = f'df_{str_df}_noleaks_pre.gzip'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/00_preprocessing/02_make_dfs/{str_filename}'
    df = pd.read_parquet(str_uri)
    list_df.append(df)
df = pd.concat(list_df)
# show
df

### Import early indicators

In [ ]:
# import early indicator (features)
print('Getting the early indicator features...')
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df_early_indicators = pd.read_csv(str_uri)
# rename
dict_rename = {
    'UniqueID': 'uniqueid',
}
df_early_indicators.rename(columns=dict_rename, inplace=True)
# show
df_early_indicators

### Get names of early indicators

In [ ]:
# get the names of the early indicators
print('Getting list of early indicators...')
str_filename = 'df_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
list_str_target = list(pd.read_csv(str_uri)['Target'])
print(f'There are {len(list_str_target)} early indicators')

### Iterate through list of early indicators and generate predictions

In [ ]:
# iterate through the list of targets
print('Iterating through early indicators...')
df_empty = pd.DataFrame({
    'uniqueid': df['uniqueid'],
    'data_set': df['data_set'],
    'target': df['target'],
})
for str_target in tqdm(list_str_target):
    # get the uniqueid and the early indicator
    list_cols = [
        'uniqueid',
        str_target,
    ]
    # subset
    df_tmp = df_early_indicators[list_cols].copy()
    # join
    df = pd.merge(
        left=df,
        right=df_tmp,
        on='uniqueid',
        how='left',
    )
    # assign early indicator as a column
    df_empty[str_target] = df[str_target]
    
    # get the model
    str_filename = 'dict_model_inference.pkl'
    str_bucket_path = f'09_early_indicators/{str_variant}/best_models/models/{str_target}/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
    cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
    os.remove(str_local_path)
    list_cols_model = list(cls_model_inference.feature_names_)
    # generate predictions
    df_empty[f'yhat_{str_target}'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df_empty

### Write to s3

In [ ]:
%%time

# write to s3
print('Writing to s3...')
str_filename = 'df_indicators_w_yhat.csv'
str_uri = f's3://{str_project}/09_early_indicators/{str_variant}/targets/{str_filename}'
df_empty.to_csv(str_uri, index=False)